# Exp11: Gemma-2 IOI, causal-subspace version
Fixes vs Exp10 (which failed its own controls):
1. **Subspace is gradient-defined** (SVD of d(logit-diff)/d(resid) at END, per prompt), not variance PCs. K sweep. **Positive control gate**: the language's own subspace must drop its own IOI, below the random subspace. If not, the subspace column says nothing.
2. **Geometry**: within-template centering (template variance removed), matched halves (equal n everywhere), split-half floor + name-set floor, plus gradient-subspace angles.
3. **Head ablation per language** (heads ranked in EN, FR, YO separately): 3x3 matrix + top-K overlap (Jaccard).

Read the 3x3 matrices together:
| World | Heads (EN heads -> FR) | Subspace (EN subspace -> FR) |
|---|---|---|
| Fully shared | drops | drops |
| Shared heads, separate subspaces | drops | no drop |
| Duplicated pathways | no drop | no drop |

Run **SMOKE=True** first.

In [1]:
import os
os.environ["http_proxy"]  = "http://proxy61.iitd.ac.in:3128"
os.environ["https_proxy"] = "http://proxy61.iitd.ac.in:3128"
os.environ["HF_TOKEN"] = os.environ.get("HF_TOKEN", "<REDACTED>")

In [2]:
import os
GPU_ID = "1"   # your sentinel; None = all visible GPUs
if GPU_ID is not None: os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID   # before torch import
import json, gc, random
import numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForCausalLM

SMOKE = True          # True: 2B only, tiny. Run first.
HF_TOKEN = os.environ.get("HF_TOKEN")
MODELS = ["google/gemma-2-2b", "google/gemma-2-9b", "google/gemma-2-27b"]
LANGS = ["en", "fr", "yo"]
LAYER_FRACS = [0.33, 0.5, 0.67, 0.85]   # activation geometry
GRAD_FRACS = [0.33, 0.5, 0.67]          # gradient-subspace geometry
CAUSAL_FRAC = 0.5                       # must be in LAYER_FRACS
K_ANGLE = 4
GATE_ACC = 0.75
POSCTRL = 0.7     # own-subspace retained logit-diff must be below this
RESULTS_DIR = "exp11_results"
os.makedirs(RESULTS_DIR, exist_ok=True)

if SMOKE:
    MODELS = MODELS[:1]; N_PROMPTS = 60; SEEDS = [0]; K_HEADS = 5; SCAN_N = 8; K_LIST = [2, 8, 32]
else:
    N_PROMPTS = 420; SEEDS = [0, 1, 2]; K_HEADS = 10; SCAN_N = 16; K_LIST = [4, 16, 64]
assert CAUSAL_FRAC in LAYER_FRACS
assert torch.cuda.is_available(), "no CUDA visible: check GPU_ID"
props = torch.cuda.get_device_properties(0)
print("SMOKE" if SMOKE else "FULL", MODELS, torch.cuda.device_count(), "GPU(s):", props.name, round(props.total_memory/2**30), "GiB")
print("27B bf16 needs ~54 GiB + activations (grad pass adds more); 9B ~18; 2B ~5")

/home/anwoy/miniconda3/envs/ke/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


SMOKE ['google/gemma-2-2b'] 1 GPU(s): NVIDIA H200 NVL 140 GiB
27B bf16 needs ~54 GiB + activations (grad pass adds more); 9B ~18; 2B ~5


In [3]:
TEMPLATES = {
 "en": ["When {N1} and {N2} went to the store, {S} gave a drink to",
        "Then, {N1} and {N2} had a lot of fun at the school. {S} gave a ball to",
        "After {N1} and {N2} went to the park, {S} handed a book to",
        "Yesterday {N1} and {N2} visited the office, and {S} sent a letter to",
        "While {N1} and {N2} were at the market, {S} offered some fruit to",
        "Today {N1} and {N2} met in the garden. Afterwards, {S} showed the photos to"],
 "fr": ["Quand {N1} et {N2} sont allés au marché, {S} a donné une pomme à",
        "Ensuite, {N1} et {N2} se sont bien amusés à l'école. {S} a donné un ballon à",
        "Après que {N1} et {N2} sont allés au parc, {S} a prêté un livre à",
        "Hier, {N1} et {N2} ont visité le bureau, et {S} a envoyé une lettre à",
        "Pendant que {N1} et {N2} étaient au marché, {S} a offert des fruits à",
        "Aujourd'hui, {N1} et {N2} se sont retrouvés dans le jardin. Ensuite, {S} a montré les photos à"],
 "yo": ["Nígbà tí {N1} àti {N2} lọ sí ọjà, {S} ra ẹ̀bùn kan fún",
        "Lẹ́yìn náà, {N1} àti {N2} gbádùn ní ilé-ẹ̀kọ́. {S} fi bọ́ọ̀lù kan fún",
        "Lẹ́yìn tí {N1} àti {N2} lọ sí ọgbà, {S} yá ìwé kan fún",
        "Lánàá {N1} àti {N2} ṣàbẹ̀wò sí ọ́fíìsì, {S} sì fi lẹ́tà kan ranṣẹ́ sí",
        "Nígbà tí {N1} àti {N2} wà ní ọjà, {S} mú èso wá fún",
        "Lónìí {N1} àti {N2} pàdé ní ọgbà. Lẹ́yìn náà, {S} fi àwòrán hàn fún"],
}
NAMES = {
 "en": "James John Robert Michael William David Richard Thomas Mary Patricia Jennifer Linda Elizabeth Barbara Susan Jessica Sarah Karen Daniel Mark Paul Steven Kevin Brian George Anna Laura Emma Alice Julia Peter Henry Frank Kate Lucy Ruth Helen Carol Diane Grace Rose Edward Andrew Joseph Charles".split(),
 "fr": "Jean Pierre Louis Jacques Michel Henri Marie Claire Sophie Camille Lucie Chloé Hugo Lucas Antoine Nicolas Alain Luc Éric Anne Isabelle Nathalie Sylvie Céline Julien Mathieu Olivier Bruno Léa Manon Étienne Gérard Vincent Émile Alexandre Margot Inès Mathilde Clémence Aurélie Baptiste Guillaume".split(),
 "yo": "Ayo Tunde Femi Bola Kemi Dayo Tola Segun Yemi Wale Funmi Ade Bisi Folake Kunle Lola Sade Tayo Seyi Bimpe Dupe Yinka Kola Ladi Remi Toyin Jide Gbenga Ife Tobi Temi Wumi Niyi Shola Sola Lekan Dele Kayode Bukola Ronke Shade Bayo Biodun Taiwo Kehinde Lanre".split(),
}
NAMES["fr"] = [n for n in NAMES["fr"] if n not in set(NAMES["en"])]   # keep lexically disjoint from EN

In [4]:
def single_token_names(tok, names):
    out, nid, seen = [], {}, set()
    for n in names:
        ids = tok.encode("a " + n, add_special_tokens=False)
        if len(ids) == 2 and ids[1] not in seen:
            out.append(n); nid[n] = ids[1]; seen.add(ids[1])
    return out, nid

def build(tok, lang, tmpl_ids, names, nid, n, rng):
    per = max(1, n // len(tmpl_ids)); batches = []
    for t in tmpl_ids:
        rows = []
        for _ in range(per):
            io, s = rng.sample(names, 2)
            n1, n2 = (io, s) if rng.random() < 0.5 else (s, io)
            ids = tok(TEMPLATES[lang][t].format(N1=n1, N2=n2, S=s))["input_ids"]
            rows.append((ids, nid[io], nid[s]))
        lens = [len(r[0]) for r in rows]
        L = max(set(lens), key=lens.count)
        rows = [r for r in rows if len(r[0]) == L]
        batches.append(dict(
            x=torch.tensor([r[0] for r in rows]),
            io=torch.tensor([r[1] for r in rows]), s=torch.tensor([r[2] for r in rows]),
            pIO=torch.tensor([r[0].index(r[1]) for r in rows]),
            pS2=torch.tensor([len(r[0]) - 1 - r[0][::-1].index(r[2]) for r in rows])))
    return batches

def slice_b(b, n): return {k: v[:n] for k, v in b.items()}

class Hooks:
    def __init__(self): self.h = []
    def add(self, m, kind, fn): self.h.append(getattr(m, kind)(fn))
    def __enter__(self): return self
    def __exit__(self, *a): [h.remove() for h in self.h]

def subspace(X, k):
    Xc = X - X.mean(0)
    return torch.linalg.svd(Xc, full_matrices=False)[2][:k]

def angle(X, Y, k):
    s = torch.linalg.svdvals(subspace(X, k) @ subspace(Y, k).T).clamp(-1, 1)
    return torch.rad2deg(torch.arccos(s)).mean().item()

In [5]:
@torch.no_grad()
def fwd(model, b, hs_layers=(), bs=32):
    dev = model.model.embed_tokens.weight.device
    cap = getattr(model.config, "final_logit_softcapping", None)
    lds, accs = [], []
    hs = {l: {"IO": [], "S2": [], "END": []} for l in hs_layers}
    for i in range(0, len(b["x"]), bs):
        x = b["x"][i:i+bs].to(dev)
        out = model.model(input_ids=x, output_hidden_states=bool(hs_layers))
        lg = model.lm_head(out.last_hidden_state[:, -1])
        if cap: lg = torch.tanh(lg / cap) * cap
        lg = lg.float(); ar = torch.arange(len(x), device=lg.device)
        d = lg[ar, b["io"][i:i+bs].to(lg.device)] - lg[ar, b["s"][i:i+bs].to(lg.device)]
        lds.append(d.cpu()); accs.append((d > 0).float().cpu())
        for l in hs_layers:
            h = out.hidden_states[l]; a2 = torch.arange(len(x), device=h.device)
            hs[l]["IO"].append(h[a2, b["pIO"][i:i+bs].to(h.device)].float().cpu())
            hs[l]["S2"].append(h[a2, b["pS2"][i:i+bs].to(h.device)].float().cpu())
            hs[l]["END"].append(h[:, -1].float().cpu())
    return torch.cat(lds), torch.cat(accs), {l: {p: torch.cat(v) for p, v in d.items()} for l, d in hs.items()}

def ev(model, batches, hs_layers=()):
    rs = [fwd(model, b, hs_layers) for b in batches]
    ld = torch.cat([r[0] for r in rs]); acc = torch.cat([r[1] for r in rs])
    hs = {l: {p: torch.cat([r[2][l][p] for r in rs]) for p in ("IO", "S2", "END")} for l in hs_layers}
    return ld, acc, hs

def head_hook(hs_idx, HD):
    def fn(mod, args):
        x = args[0].clone()
        for h in hs_idx: x[..., h*HD:(h+1)*HD] = 0
        return (x,) + tuple(args[1:])
    return fn

def ev_heads(model, batches, heads):   # heads: list of (layer, head)
    HD = model.config.head_dim; by = {}
    for l, h in heads: by.setdefault(l, []).append(h)
    with Hooks() as H:
        for l, hh in by.items():
            H.add(model.model.layers[l].self_attn.o_proj, "register_forward_pre_hook", head_hook(hh, HD))
        return ev(model, batches)[0]

def sub_hook(V, mu):
    def fn(mod, args, out):
        tup = isinstance(out, tuple); h = (out[0] if tup else out).clone()
        last = h[:, -1].float(); Vd = V.to(h.device); m = mu.to(h.device)
        h[:, -1] = (last - ((last - m) @ Vd.T) @ Vd).to(h.dtype)
        return (h,) + tuple(out[1:]) if tup else h
    return fn

def ev_sub(model, batches, layer_idx, V, mu):
    with Hooks() as H:
        H.add(model.model.layers[layer_idx], "register_forward_hook", sub_hook(V, mu))
        return ev(model, batches)[0]

In [6]:
def ev_hs(model, batches, layers):
    rs = [fwd(model, b, layers) for b in batches]
    hs = {l: {p: torch.cat([r[2][l][p] for r in rs]) for p in ("IO", "S2", "END")} for l in layers}
    tid = torch.cat([torch.full((len(b["x"]),), t) for t, b in enumerate(batches)])
    return hs, tid

def grad_END(model, b_list, layers, bs=16):
    """per-prompt d(logit_diff)/d(resid at END) for each layer in `layers` (hidden_states index)."""
    dev = model.model.embed_tokens.weight.device
    cap = getattr(model.config, "final_logit_softcapping", None)
    G = {l: [] for l in layers}; tid = []
    for t, b in enumerate(b_list):
        for i in range(0, len(b["x"]), bs):
            x = b["x"][i:i+bs].to(dev)
            with torch.no_grad(): emb = model.model.embed_tokens(x)
            emb = emb.detach().requires_grad_(True)
            with torch.enable_grad():
                out = model.model(inputs_embeds=emb, output_hidden_states=True)
                for l in layers: out.hidden_states[l].retain_grad()
                lg = model.lm_head(out.last_hidden_state[:, -1])
                if cap: lg = torch.tanh(lg / cap) * cap
                lg = lg.float(); ar = torch.arange(len(x), device=lg.device)
                d = lg[ar, b["io"][i:i+bs].to(lg.device)] - lg[ar, b["s"][i:i+bs].to(lg.device)]
                d.sum().backward()
            for l in layers: G[l].append(out.hidden_states[l].grad[:, -1].float().cpu())
            tid.append(torch.full((len(x),), t))
    return {l: torch.cat(v) for l, v in G.items()}, torch.cat(tid)

def center_by(X, tid):
    X = X.clone()
    for t in tid.unique():
        m = tid == t; X[m] = X[m] - X[m].mean(0)
    return X

def subspace(X, k, center=True):
    X = X.float(); X = X - X.mean(0) if center else X
    return torch.linalg.svd(X, full_matrices=False)[2][:k]

def angle(X, Y, k, center=True):
    s = torch.linalg.svdvals(subspace(X, k, center) @ subspace(Y, k, center).T).clamp(-1, 1)
    return torch.rad2deg(torch.arccos(s)).mean().item()

def shuf(Z, g): return Z[torch.randperm(len(Z), generator=g)]

In [7]:
def run_model(name):
    path = f"{RESULTS_DIR}/{name.split('/')[-1]}{'_smoke' if SMOKE else ''}.json"
    if os.path.exists(path):
        print("cached", path); return json.load(open(path))
    tok = AutoTokenizer.from_pretrained(name, token=HF_TOKEN)
    model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.bfloat16, device_map="auto",
                                                 attn_implementation="eager", token=HF_TOKEN).eval()
    for p in model.parameters(): p.requires_grad_(False)
    NL, NH = model.config.num_hidden_layers, model.config.num_attention_heads
    res = {"model": name, "gate": {}, "angles": [], "heads": {}, "subspace": {}, "posctrl": {}}

    names, nid = {}, {}
    for L in LANGS:
        names[L], nid[L] = single_token_names(tok, NAMES[L])
        print(L, len(names[L]), "single-token names")
        if len(names[L]) < 8: print("  WARNING few names for", L)
    nid_all = {n: i for L in LANGS for n, i in nid[L].items()}
    en = names["en"][:]; random.Random(123).shuffle(en)
    nA, nB = en[:len(en)//2], en[len(en)//2:]
    T = list(range(6))

    def conds(seed):
        r = random.Random(seed)
        return {"EN": build(tok, "en", T, names["en"], nid_all, N_PROMPTS, r),
                "FR": build(tok, "fr", T, names["fr"], nid_all, N_PROMPTS, r),
                "YO": build(tok, "yo", T, names["yo"], nid_all, N_PROMPTS, r),
                "EN_NA": build(tok, "en", T, nA, nid_all, N_PROMPTS, r),
                "EN_NB": build(tok, "en", T, nB, nid_all, N_PROMPTS, r)}

    # 1. gate
    C0 = conds(SEEDS[0]); ok = {}
    for L in LANGS:
        ld, acc, _ = ev(model, C0[L.upper()])
        res["gate"][L] = {"acc": acc.mean().item(), "logit_diff": ld.mean().item(), "n": len(ld)}
        ok[L] = acc.mean().item() >= GATE_ACC
    print("GATE", {k: round(v["acc"], 3) for k, v in res["gate"].items()}, "ok:", ok)
    keys = [k for k in C0 if not (k in ("FR", "YO") and not ok[k.lower()])]

    layers = [int(f * NL) for f in LAYER_FRACS]
    glayers = sorted(set(int(f * NL) for f in GRAD_FRACS + [CAUSAL_FRAC]))
    Lh = int(CAUSAL_FRAC * NL)

    # 2. geometry: activations (within-template centered) + gradient subspaces; matched halves + floors
    for seed in SEEDS:
        C = C0 if seed == SEEDS[0] else conds(seed)
        g = torch.Generator().manual_seed(seed)
        H, TI, GR = {}, {}, {}
        for k in keys:
            H[k], TI[k] = ev_hs(model, C[k], layers)
            GR[k] = grad_END(model, C[k], glayers)[0]
        if seed == SEEDS[0]: H0, GR0 = H, GR

        def rows_for(getX, center, pos, l, frac):
            X = {k: getX(k) for k in keys}
            e = shuf(X["EN"], g)
            n = min(len(e) // 2, *[len(X[k]) for k in keys])
            row = {"seed": seed, "frac": frac, "pos": pos,
                   "floor_split": angle(e[:n], e[n:2*n], K_ANGLE, center),
                   "floor_nameset": angle(shuf(X["EN_NA"], g)[:n], shuf(X["EN_NB"], g)[:n], K_ANGLE, center)}
            for L in ("fr", "yo"):
                row["EN-" + L.upper()] = angle(e[:n], shuf(X[L.upper()], g)[:n], K_ANGLE, center) if ok[L] else None
            row["excess_FR"] = None if row["EN-FR"] is None else row["EN-FR"] - max(row["floor_split"], row["floor_nameset"])
            row["excess_YO"] = None if row["EN-YO"] is None else row["EN-YO"] - max(row["floor_split"], row["floor_nameset"])
            res["angles"].append(row)

        for f, l in zip(LAYER_FRACS, layers):
            for p in ("IO", "S2", "END"):
                rows_for(lambda k: center_by(H[k][l][p], TI[k]), True, p, l, f)
        for f in GRAD_FRACS:
            l = int(f * NL)
            rows_for(lambda k: GR[k][l], False, "GRAD", l, f)
        del H, GR

    # 3a. heads: rank per language, ablate each language's top-K in every language
    tops = {}
    for L in LANGS:
        if not ok[L]: continue
        sb = slice_b(C0[L.upper()][0], SCAN_N); base = ev(model, [sb])[0].mean().item()
        sc = np.zeros((NL, NH))
        for l in range(NL):
            for h in range(NH):
                sc[l, h] = base - ev_heads(model, [sb], [(l, h)]).mean().item()
        flat = sorted(((sc[l, h], l, h) for l in range(NL) for h in range(NH)), reverse=True)
        tops[L] = [(l, h) for _, l, h in flat[:K_HEADS]]
    rr = random.Random(0); allh = [(l, h) for l in range(NL) for h in range(NH)]
    rands = [rr.sample(allh, K_HEADS) for _ in range(3)]
    res["heads"] = {"top": tops, "jaccard": {}, "retained": {}, "retained_rand": {}}
    for a in tops:
        for b in tops:
            if a < b: res["heads"]["jaccard"][f"{a}-{b}"] = len(set(tops[a]) & set(tops[b])) / len(set(tops[a]) | set(tops[b]))
    for tgt in tops:
        clean = ev(model, C0[tgt.upper()])[0].mean().item()
        res["heads"]["retained"][tgt] = {src: ev_heads(model, C0[tgt.upper()], tops[src]).mean().item() / clean for src in tops}
        res["heads"]["retained_rand"][tgt] = float(np.mean([ev_heads(model, C0[tgt.upper()], r).mean().item() for r in rands])) / clean

    # 3b. gradient-subspace ablation: mean-ablate src subspace in tgt (mean = tgt END mean at layer Lh)
    mus = {L: H0[L.upper()][Lh]["END"].mean(0) for L in LANGS if ok[L]}
    cleans = {L: ev(model, C0[L.upper()])[0].mean().item() for L in mus}
    gg = torch.Generator().manual_seed(0); D = mus["en"].shape[0]
    for K in K_LIST:
        Kc = min(K, len(GR0["EN"][Lh]))
        Vr = torch.linalg.qr(torch.randn(D, Kc, generator=gg))[0].T
        res["subspace"][K] = {}
        for src in mus:
            V = subspace(GR0[src.upper()][Lh], Kc, center=False); res["subspace"][K][src] = {}
            for tgt in mus:
                a = ev_sub(model, C0[tgt.upper()], Lh - 1, V, mus[tgt]).mean().item()
                r = ev_sub(model, C0[tgt.upper()], Lh - 1, Vr, mus[tgt]).mean().item()
                res["subspace"][K][src][tgt] = {"retained": a / cleans[tgt], "retained_rand": r / cleans[tgt]}
        res["posctrl"][K] = {L: {"own": res["subspace"][K][L][L]["retained"], "rand": res["subspace"][K][L][L]["retained_rand"],
                                 "pass": res["subspace"][K][L][L]["retained"] < POSCTRL} for L in mus}
    print("POSITIVE CONTROL (own subspace retained; need < %.2f):" % POSCTRL, res["posctrl"])

    json.dump(res, open(path, "w"), indent=1)
    del model; gc.collect(); torch.cuda.empty_cache()
    return res

In [8]:
ALL = {m: run_model(m) for m in MODELS}

for m, r in ALL.items():
    print("\n=====", m)
    print("GATE:", {k: round(v["acc"], 3) for k, v in r["gate"].items()})
    print("\nGEOMETRY (deg; matched halves; excess = cross - max(floors)); GRAD = gradient subspaces, others = within-template-centered activations")
    cols = ["floor_split", "floor_nameset", "EN-FR", "EN-YO", "excess_FR", "excess_YO"]
    df = pd.DataFrame(r["angles"]).groupby(["frac", "pos"])[cols].agg(["mean", "std"])
    print(df.round(1).to_string())
    h = r["heads"]
    print("\nHEAD ABLATION retained fraction (rows=tgt lang, cols=heads ranked on src lang):")
    print(pd.DataFrame(h["retained"]).T.round(3).to_string())
    print("random heads:", {k: round(v, 3) for k, v in h["retained_rand"].items()}, " top-K Jaccard:", {k: round(v, 2) for k, v in h["jaccard"].items()})
    for K, mat in r["subspace"].items():
        print(f"\nGRAD-SUBSPACE ABLATION K={K} retained (rows=src subspace, cols=tgt):")
        print(pd.DataFrame({s: {t: x["retained"] for t, x in d.items()} for s, d in mat.items()}).T.round(3).to_string())
        print("  random-subspace:", {t: round(x["retained_rand"], 3) for t, x in mat["en"].items()})
        for tgt in mat:
            own = 1 - mat[tgt][tgt]["retained"]
            if own > 0.2:
                print(f"  tgt={tgt}: cross-effect / own-effect:", {s: round((1 - mat[s][tgt]["retained"]) / own, 2) for s in mat if s != tgt})
    print("\nPOSITIVE CONTROL:", r["posctrl"])

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 288/288 [00:00<00:00, 686.15it/s]


en 45 single-token names
fr 34 single-token names
yo 12 single-token names
GATE {'en': 1.0, 'fr': 1.0, 'yo': 0.85} ok: {'en': True, 'fr': True, 'yo': True}
POSITIVE CONTROL (own subspace retained; need < 0.70): {2: {'en': {'own': 1.0288461761213241, 'rand': 0.9994939348631968, 'pass': False}, 'fr': {'own': 1.0085106300088065, 'rand': 1.0026595338322348, 'pass': False}, 'yo': {'own': 0.9952393667558121, 'rand': 0.9933879723896226, 'pass': False}}, 8: {'en': {'own': 1.097166012067725, 'rand': 1.0005060651368032, 'pass': False}, 'fr': {'own': 1.0797871715507676, 'rand': 1.0, 'pass': False}, 'yo': {'own': 1.0674425363424829, 'rand': 0.9984131424301828, 'pass': False}}, 32: {'en': {'own': 1.101720598298954, 'rand': 1.0015181954104098, 'pass': False}, 'fr': {'own': 1.0872339880266801, 'rand': 1.0015957202993409, 'pass': False}, 'yo': {'own': 1.0766992660344863, 'rand': 0.9973551768489018, 'pass': False}}}

===== google/gemma-2-2b
GATE: {'en': 1.0, 'fr': 1.0, 'yo': 0.85}

GEOMETRY (deg; match